# Bispectrum Estimator Examples

This notebook focuses on `BispectrumMultipoles`. It uses the same helper names and backend pattern as `estimators_examples.ipynb`, then goes directly to Scoccimarro and Sugiyama bispectrum calculations.

## Setup

In [ ]:
import numpy as np

%matplotlib inline
%config InlineBackend.figure_format = "retina"

from helpers import load_estimator_parameters, make_lagrangian_mock

from acm import setup_logging
from acm.estimators.galaxy_clustering.backends.jaxpower import (
    JaxpowerBackend,  # noqa: F401 - register backend
)
from acm.estimators.galaxy_clustering.bispectrum import BispectrumMultipoles

setup_logging()

In [ ]:
# Custom helper function to summarize the bispectrum result
def summarize_bispectrum(result, ell=None, nrows=5) -> None:  # noqa: ANN001
    """Print a small summary of a Mesh3SpectrumPoles result."""
    ell = result.ells[0] if ell is None else ell
    pole = result.get(ell)
    k = np.asarray(pole.coords("k"))

    print(type(result))
    print(f"basis: {result.basis}")
    print(f"ells: {result.ells}")
    print(f"number of bins: {len(k)}")
    print("first k coordinates:")
    print(k[:nrows])

## Build Mock And Backend

In [ ]:
los = "z"
data_positions, boxsize = make_lagrangian_mock(boxsize=500.0, los=los)
params = load_estimator_parameters("bispectrum")

estimator = BispectrumMultipoles(
    backend="jaxpower",
    data_positions=data_positions,
    boxsize=boxsize,
    **params["initialization"],
)
estimator.backend.set_density_contrast(
    resampler="tsc",
    interlacing=3,
    compensate=True,
)

print(f"Number of mock particles: {len(data_positions)}")
print(f"Box size: {boxsize:.1f} Mpc/h")
print(f"Mesh size: {estimator.backend.meshsize}")

## Scoccimarro Multipoles

The default Scoccimarro basis computes `(0, 2)` multipoles. The plot helper uses triangle-bin index and scales by `k1 k2 k3` by default.

In [ ]:
result = estimator.compute(los=los, basis="scoccimarro", **params["compute"])
summarize_bispectrum(result)

fig, ax = BispectrumMultipoles.plot(result, ells=(0, 2))
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()

## Diagonal Sugiyama Multipoles

The diagonal Sugiyama basis is requested at compute time. For diagonal Sugiyama results, the plot helper uses physical `k` on the x-axis and scales by `k^2`.

In [ ]:
sugiyama_result = estimator.compute(
    basis="sugiyama-diagonal",
    los=los,
    ells=[(0, 0, 0), (2, 0, 2)],
    **params["compute"],
)
summarize_bispectrum(sugiyama_result, ell=(0, 0, 0))

fig, ax = BispectrumMultipoles.plot(
    sugiyama_result,
    ells=[(0, 0, 0), (2, 0, 2)],
)
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
